# Python OOP Deep Dive: Lead Data Engineer Edition

**Target Audience**: Senior/Staff Data Engineers & Software Engineers.
**Goal**: Move beyond basic syntax. Understand *why* and *how* to use advanced OOP patterns in scalable data systems.

## Agenda
1.  **Essential Magic Methods**: Not just syntax, but *architectural* tools for cleaner pipelines.
2.  **Object Creation Internals**: `__new__` vs `__init__` (Singletons, Metaclasses).
3.  **Design Patterns**: Mixins & Composition over Inheritance.
4.  **Advanced Internals**: Descriptors, MRO, and Attribute Access.
5.  **Interview Gauntlet**: Tricky questions that test deep understanding.

---

## 1. Essential Magic Methods (The "Architectural" Ones)

In a Lead role, you use these to make your libraries (e.g., internal ETL frameworks) intuitive for other engineers.

### 1.1 `__repr__` vs `__str__`: Observability

**Real-World Use Case**: 
When debugging distributed systems (Spark, Celery, Airflow), logs are your lifeline. A default object print `<Object at 0x123>` is useless in a log file. 
- **`__repr__`**: Unambiguous representation. Crucial for logs/debugging. Should ideally be valid Python code to recreate the object.
- **`__str__`**: Readable representation for end-users (e.g., CLI output).

In [ ]:
class DatabaseConfig:
    def __init__(self, host, port, schema):
        self.host = host
        self.port = port
        self.schema = schema
    
    def __repr__(self):
        # Lead Eng Tip: Include all fields needed to debug connection issues
        return f"DatabaseConfig(host='{self.host}', port={self.port}, schema='{self.schema}')"
    
    def __str__(self):
        # User friendly: Hide sensitive info or simplify
        return f"DB Connection ({self.host}:{self.port})"

config = DatabaseConfig("10.0.0.1", 5432, "warehouse")

print(f"Log (repr): {repr(config)}")  # What you see in Datadog/CloudWatch
print(f"UI (str): {str(config)}")     # What you show in a CLI tool

### 1.2 `__getitem__` & `__len__`: Custom Datasets

**Real-World Use Case**:
Creating custom datasets for Machine Learning (PyTorch/TensorFlow) or batch processing iterators. You want your custom data loader to behave exactly like a Python list.

In [ ]:
class BatchLoader:
    """Simulates a lazy-loading dataset wrapper."""
    def __init__(self, data_files):
        self.files = data_files
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        # Real-world: This might read a Parquet file from S3
        file_path = self.files[idx]
        print(f"[IO] Loading batch from {file_path}...")
        return f"Data({file_path})"

loader = BatchLoader(["part-001.parquet", "part-002.parquet", "part-003.parquet"])

# It behaves just like a list!
print(f"Total batches: {len(loader)}")
first_batch = loader[0]  # Triggers the custom logic

### 1.3 `__call__`: Stateful Decorators & UDFs

**Real-World Use Case**:
1.  **Spark UDFs**: Sometimes you need a UDF that maintains state (e.g., loading a heavy model once). A class with `__call__` is cleaner than a closure.
2.  **Decorators**: Creating parameterized decorators (e.g., `@retry(attempts=3)`).

In [ ]:
class ModelPredictor:
    """Stateful UDF example."""
    def __init__(self, model_path):
        self.model_path = model_path
        self._model = None
    
    def _load_model(self):
        if not self._model:
            print(f"[Heavy Operation] Loading model from {self.model_path}...")
            self._model = "LoadedModel"
    
    def __call__(self, input_data):
        # This makes the instance callable like a function: predictor(data)
        self._load_model()
        return f"Prediction for {input_data} using {self._model}"

# Usage in a pipeline
predictor = ModelPredictor("s3://bucket/model.pkl")

# First call loads model
print(predictor("row1"))
# Second call reuses loaded model
print(predictor("row2"))

---

## 2. Object Creation Internals (`__new__` vs `__init__`)

This is a favorite topic for Staff Engineer interviews to test depth.

### The Concept
- **`__new__(cls, ...)`**: The **Allocator**. It creates the object in memory. It returns the instance.
- **`__init__(self, ...)`**: The **Initializer**. It populates the object. It returns `None`.

**Real-World Use Case**: 
1.  **Singleton Pattern**: Ensuring only one Database Connection Pool exists.
2.  **Immutable Types**: Subclassing `str` or `tuple` (since they can't be modified in `__init__`).

In [ ]:
class DBConnectionPool:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        # Hook into creation to check if instance already exists
        if cls._instance is None:
            print("Creating NEW Connection Pool...")
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self, connection_string):
        # CAUTION: __init__ runs every time you call the class!
        # You usually need a guard clause here too for Singletons.
        self.conn_str = connection_string

pool1 = DBConnectionPool("postgres://primary")
pool2 = DBConnectionPool("postgres://secondary")

print(f"Same object? {pool1 is pool2}")
print(f"Pool1 String: {pool1.conn_str}") # Note: It got overwritten by the second call's init!

---

## 3. Mixins & Composition (Lead Engineer Patterns)

**The Problem**: Inheritance hierarchies get messy. "Is a DatabaseLoader also a Logger? And a Validator?"
**The Solution**: Mixins. Small, focused classes that provide specific behavior. 

**Real-World Use Case**:
Building a modular ETL framework. You have `S3Extractor`, `PostgresExtractor`, `ApiExtractor`. All of them need Logging and Validation capabilities.

In [ ]:
class LoggingMixin:
    """Provides standard logging capability."""
    def log_info(self, msg):
        print(f"[INFO] {self.__class__.__name__}: {msg}")

class S3AccessMixin:
    """Provides S3 connectivity."""
    def read_s3(self, path):
        self.log_info(f"Reading from S3: {path}")
        return "data"

class PostgresLoader(LoggingMixin, S3AccessMixin):
    """
    Composed of multiple behaviors.
    We didn't have to inherit from a giant 'BaseETL' class.
    """
    def run(self, s3_path):
        data = self.read_s3(s3_path)
        self.log_info(f"Loading {data} into Postgres")

loader = PostgresLoader()
loader.run("s3://bucket/data.csv")

---

## 4. Advanced Internals: Descriptors & Attribute Access

### 4.1 Descriptors (`__get__`, `__set__`)
**Real-World Use Case**: 
Strict type validation in Data Models. If you are building an ORM or a config validator, you use descriptors. This is how `@property` works under the hood.

In [ ]:
class IntField:
    """Descriptor to enforce integer types."""
    def __init__(self, name):
        self.name = name
    
    def __get__(self, instance, owner):
        return instance.__dict__.get(self.name)
    
    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} must be an integer")
        instance.__dict__[self.name] = value

class TableSchema:
    row_count = IntField("row_count")
    
    def __init__(self, count):
        self.row_count = count

schema = TableSchema(100)
print(f"Row count: {schema.row_count}")

try:
    schema.row_count = "many"  # Raises TypeError
except TypeError as e:
    print(f"Validation Error: {e}")

### 4.2 `__getattr__` vs `__getattribute__`

**Real-World Use Case**: 
Creating a "Wrapper" or "Proxy" class. For example, wrapping a Spark DataFrame to add custom metadata methods while delegating everything else to the original DataFrame.

- **`__getattr__`**: Called ONLY if attribute is NOT found. Safe for delegation.
- **`__getattribute__`**: Called for EVERY access. Dangerous (infinite recursion risk).

In [ ]:
class DataFrameWrapper:
    def __init__(self, df):
        self._df = df
    
    def custom_method(self):
        return "Custom Logic"
    
    def __getattr__(self, name):
        # Delegate unknown calls to the underlying dataframe
        print(f"[Delegation] Accessing {name} on inner DF")
        return getattr(self._df, name)

# Mocking a DataFrame for demo
class MockDF:
    def count(self): return 1000

wrapper = DataFrameWrapper(MockDF())
print(wrapper.custom_method())  # Hits Wrapper
print(wrapper.count())          # Hits __getattr__ -> MockDF

---

## 5. Interview Gauntlet (Tricky Questions)

**Q1: What happens if `__new__` returns an object of a different class?**
- **Answer**: `__init__` is skipped! Python only calls `__init__` if `__new__` returns an instance of the *same* class.

In [ ]:
class A:
    def __new__(cls):
        return list()  # Returning a list instead of A
    
    def __init__(self):
        print("Init A")  # This will NEVER run

obj = A()
print(f"Object type: {type(obj)}")

**Q2: How does MRO (Method Resolution Order) work in the Diamond Problem?**
- **Answer**: Python uses the C3 Linearization algorithm. It searches depth-first, left-to-right, but ensures no class is visited before its subclasses.

In [ ]:
class Base:
    def greet(self): print("Base")

class Left(Base):
    def greet(self): 
        print("Left")
        super().greet()

class Right(Base):
    def greet(self): 
        print("Right")
        super().greet()

class Child(Left, Right):
    def greet(self): 
        print("Child")
        super().greet()

print(Child.mro())
# Output will be Child -> Left -> Right -> Base
# Notice Right is called AFTER Left, even though Left inherits from Base.
c = Child()
c.greet()